In [19]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import re
import nltk
from import_dataset import unlabelledTxtToDataFrame
from preprocessing import TextPreprocessor
from nltk.stem import SnowballStemmer, WordNetLemmatizer

random_state=42
np.random.seed(random_state)

base = Path.cwd().parent
data_path= base / "data" / "1_input"

# custom nltk directory
NLTK_DATA = Path.home() / "nltk_data"
nltk.data.path.append(str(NLTK_DATA))

RESOURCES = [
    "punkt",
    "stopwords",
    "averaged_perceptron_tagger",
    "wordnet"
]

for res in RESOURCES:
    try:
        nltk.data.find(res)
    except LookupError:
        nltk.download(res, download_dir=str(NLTK_DATA), quiet=True)

tokenizer = nltk.word_tokenize
stop_words = set(nltk.corpus.stopwords.words("english"))
stemmer = SnowballStemmer("english")
lemmatizer = WordNetLemmatizer()

In [20]:
loader = unlabelledTxtToDataFrame(random_state=random_state)
df_train = loader.loadFromFolder(
    root= data_path,
    dataset_type="train",
    label_map= {
        "pos": 1,
        "neg": 0
    },
    threshold = 500,
    subsample =1250
)

Processing folder: c:\Users\marco\OneDrive\Desktop\nlp-classification\data\1_input\train\pos
Processing folder: c:\Users\marco\OneDrive\Desktop\nlp-classification\data\1_input\train\neg
entrato primo if


In [21]:
df_copy = df_train.copy()

In [22]:
tp = TextPreprocessor(df_copy, 
                      'text', 
                      stop_words = stop_words, 
                      tokenizer = False, 
                      stemmer = None, 
                      lemmatizer = lemmatizer)

In [23]:
df = tp.process(tokenize = True, 
                remove_stop =True, 
                method ='lemm')

In [24]:
df.tail(10)

,text,target
2490,"[movie, extra, long, tale, classic, novel, com...",0
2491,"[could, cute, movie, kid, grandson, watch, wat...",0
2492,"[movie, receive, lot, bad, press, people, unde...",0
2493,"[well, contrast, comment, previously, write, s...",0
2494,"[wow, bad, movie, ever, reason, sign, imdb, co...",0
2495,"[film, waste, time, even, rent, dvd, super, sp...",0
2496,"[feel, like, watch, snuff, film, beautifully, ...",0
2497,"[attempted, watch, movie, twice, even, fast, f...",0
2498,"[start, movie, soon, become, aware, name, film...",0
2499,"[first, minute, tinseltown, finger, teeter, re...",0


In [25]:
df_train.tail(10)

,text,target
2490,The movie is an extra-long tale of a classic n...,0
2491,This could be a cute movie for kids My grandso...,0
2492,This movie has received a lot of bad press fro...,0
2493,Well.......in contrast to other comments previ...,0
2494,wow this is the worst movie ever. the only rea...,0
2495,"This film was a waste of time, even rented on ...",0
2496,I feel like I've just watched a snuff film.......,0
2497,I attempted watching this movie twice and even...,0
2498,from the start of this movie you soon become a...,0
2499,The first 30 minutes of Tinseltown had my fing...,0


### Attention: should we apply word preprocessing to pos_words and neg_words as well?

In [26]:
pos_words = [
    "good","great","excellent","amazing","fantastic","love","wonderful","best","awesome","positive",
    "beautiful","brilliant","cool","delightful","enjoyable","fun","funny","glad","happy","impressive",
    "incredible","nice","perfect","pleasant","super","terrific","outstanding","superb","marvelous","fabulous",
    "favorite","charming","smart","clever","fresh","exciting","engaging","entertaining","moving","touching",
    "heartwarming","inspiring","satisfying","solid","strong","powerful","creative","original","unique","memorable",
    "masterpiece","phenomenal","spectacular","stunning","lovely","sweet","adorable","bright","lively","vibrant",
    "smooth","fast","easy","clear","helpful","useful","reliable","worth","worthy","recommend",
    "enjoy","enjoyed","enjoying","loved","likes","liked","winning","wins","success","successful",
    "improved","improvement","perfectly","works","worked","working","effective","efficient","flawless","comfortable",
    "beautifully","elegant","epic","legendary","amaze","amazed","amazing","graceful","quality","premium",
    "top","topnotch","bestever","favorite","gold","pleasure","delicious","friendly","kind","supportive"
]

neg_words = [
    "bad","pity","terrible","awful","horrible","worst","hate","disgusting","negative",
    "boring","dull","slow","annoying","annoyed","annoyance","disappointed","disappointing","disappoint","sad",
    "poor","weak","ugly","messy","confusing","complicated","hard","difficult","broken","useless",
    "waste","wasted","wasting","problem","problems","issue","issues","bug","bugs","error",
    "fault","faulty","fail","failed","failing","failure","crash","crashes","crashed","lag",
    "cheap","fake","ridiculous","stupid","dumb","nonsense","mediocre","predictable","forgettable","unpleasant",
    "uncomfortable","painful","noisy","dirty","smelly","gross","pathetic","lame","weakest","inferior",
    "frustrating","frustrated","frustrate","hate","hated","hates","angry","mad","upset","regret",
    "regrettable","dislike","disliked","cry","crying","sadly","depressing","depressed","stress","stressful",
    "scam","ripoff","overpriced","expensive","worthless","terribly","poorly","unfair","unhappy","negative",
    "garbage","trash","disaster","catastrophe","horrendous","atrocious","abysmal","shame","shameful","unwatchable",
    "unusable","unreliable","inconsistent","slowly","glitch","glitches","missing","lack","lacking","sucks"
]



In [27]:
def sentiment_score(df, column, pos_words, neg_words):

    def score_tokens(tokens):
        pos = sum(1 for w in tokens if w in pos_words)
        neg = sum(1 for w in tokens if w in neg_words)
        return pos - neg

    return df[column].map(score_tokens)

In [28]:
df["sentiment_score"] = sentiment_score(df, "text", pos_words, neg_words)

In [29]:
df[df['sentiment_score']<1]

,text,target,sentiment_score
13,"[unlikely, duo, zero, mostel, harry, belafonte...",1,-1
27,"[last, hunt, forget, hollywood, classic, weste...",1,-1
36,"[pleasure, watch, short, film, cure, first, ti...",1,-1
43,"[superb, initially, think, give, amrita, prita...",1,0
48,"[imagination, terrible, thing, waste, especial...",1,-6
...,...,...,...
2491,"[could, cute, movie, kid, grandson, watch, wat...",0,0
2494,"[wow, bad, movie, ever, reason, sign, imdb, co...",0,-6
2496,"[feel, like, watch, snuff, film, beautifully, ...",0,0
2498,"[start, movie, soon, become, aware, name, film...",0,-1
